<a id="ohm"></a>

$$
\text{欧姆定律 (Ohm's Law)} \\
\text{电流 (Current): 单位时间流过截面的电荷量 } I = \frac{dq}{dt} \text{ (A = C/s)} \\
\text{电压 (Voltage): 两点之间单位电荷的电势能差 } V = \frac{W}{q} \text{ (V = J/C)} \\
\text{欧姆定律: 对线性电阻, 电流与电压成正比 } V = I R \text{, R 是电阻 (Resistance, } \Omega \text{), } G = \frac{1}{R} \text{ 是电导 (Conductance, S)} \\
\text{电阻由材料和形状决定: } R = \rho \frac{L}{A} \text{ (} \rho \text{ 电阻率, L 长度, A 截面积)} \\ \quad \\
\text{功率 (Power) 的推导: 电荷 dq 从高电势移到低电势, 释放能量 } dW = V \, dq \\
P = \frac{dW}{dt} = V \frac{dq}{dt} = V I \xrightarrow{V = IR} P = I^2 R = \frac{V^2}{R}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

R = 100.0
V = np.linspace(0, 12, 7)
I = V / R
print(f'电压 = {V}')
print(f'电流 = {I}')
P_vi, P_i2r, P_v2r = V * I, I**2 * R, V**2 / R
print(f'VI == I^2 R == V^2 / R ? {np.allclose(P_vi, P_i2r) and np.allclose(P_vi, P_v2r)}')

# 铜导线: rho = 1.68e-8 ohm m, 长 10 m, 截面 1 mm^2
rho, L_wire, A_wire = 1.68e-8, 10.0, 1e-6
print(f'10 m 铜导线 (1 mm^2) 的电阻 = {rho * L_wire / A_wire:.4f} ohm')

<a id="kcl-kvl"></a>

$$
\text{基尔霍夫定律 (Kirchhoff's Laws)} \\ \quad \\
\text{1. 基尔霍夫电流定律 (KCL): 流入一个节点的电流之和 = 流出的电流之和, } \sum_k I_k = 0 \\
\text{推导: 电荷守恒 (Conservation of Charge). 节点 (导线的交汇点) 不能储存电荷, 节点电荷 } q_{node} \text{ 不变} \\
\frac{dq_{node}}{dt} = \sum I_{in} - \sum I_{out} = 0 \\ \quad \\
\text{2. 基尔霍夫电压定律 (KVL): 沿任意闭合回路, 电压升降之和为 0, } \sum_k V_k = 0 \\
\text{推导: 能量守恒. 静电场是保守场 (Conservative Field), 每个点有唯一的电势 } \phi \\
\text{沿回路 } a \to b \to c \to a \text{: } (\phi_a - \phi_b) + (\phi_b - \phi_c) + (\phi_c - \phi_a) = 0 \\
\text{(如果不为 0, 电荷绕一圈就能凭空获得能量)}
$$

<a id="series-parallel"></a>

$$
\text{串联与并联 (Series and Parallel): 由 KCL / KVL 推导等效电阻} \\ \quad \\
\text{串联: 只有一条路径, 由 KCL 每个电阻的电流相同, 由 KVL 总电压 = 各电阻电压之和} \\
V = I R_1 + I R_2 + \cdots + I R_n = I (R_1 + \cdots + R_n) \implies R_{eq} = \sum_k R_k \\
\text{分压公式 (Voltage Divider): } V_k = I R_k = \frac{R_k}{\sum R} V \\ \quad \\
\text{并联: 两端接在同两个节点上, 由 KVL 每个电阻的电压相同, 由 KCL 总电流 = 各支路电流之和} \\
I = \frac{V}{R_1} + \frac{V}{R_2} + \cdots = V \sum_k \frac{1}{R_k} \implies \frac{1}{R_{eq}} = \sum_k \frac{1}{R_k} \quad (\text{电导相加 } G_{eq} = \sum G_k) \\
\text{两个电阻并联: } R_{eq} = \frac{R_1 R_2}{R_1 + R_2} \text{ (记作 } R_1 \parallel R_2 \text{), 并联后比最小的那个电阻还小} \\
\text{分流公式 (Current Divider): } I_1 = \frac{V}{R_1} = \frac{I R_{eq}}{R_1} = \frac{R_2}{R_1 + R_2} I \text{ (电流更多地走电阻小的支路)}
$$

In [ ]:
def series(*rs):
    return sum(rs)

def parallel(*rs):
    return 1 / sum(1 / r for r in rs)

R1, R2, R3 = 100.0, 220.0, 470.0
V_s = 12.0

# 串联: 分压
I_series = V_s / series(R1, R2, R3)
V_drops = I_series * np.array([R1, R2, R3])
print(f'串联: R_eq = {series(R1, R2, R3)}, I = {I_series * 1000:.4f} mA, 各电压 = {V_drops}, 和 = {V_drops.sum():.4f} (KVL)')

# 并联: 分流
I_branch = V_s / np.array([R1, R2, R3])
print(f'并联: R_eq = {parallel(R1, R2, R3):.4f} (< 最小电阻 {R1}), 支路电流 = {I_branch * 1000} mA, '
      f'和 = {I_branch.sum() * 1000:.4f} mA == V / R_eq = {V_s / parallel(R1, R2, R3) * 1000:.4f} mA (KCL)')
I_total = 0.05
print(f'分流公式: I1 = R2/(R1+R2) I = {R2 / (R1 + R2) * I_total * 1000:.4f} mA, 直接算 V/R1 = {I_total * parallel(R1, R2) / R1 * 1000:.4f} mA')

# 混联: R1 串联 (R2 并 R3)
R_mix = series(R1, parallel(R2, R3))
I_mix = V_s / R_mix
V_parallel = I_mix * parallel(R2, R3)
print(f'混联: R_eq = {R_mix:.4f}, I = {I_mix * 1000:.4f} mA, 并联部分电压 = {V_parallel:.4f} V, '
      f'流过 R2 和 R3 = {V_parallel / R2 * 1000:.4f} + {V_parallel / R3 * 1000:.4f} = {(V_parallel / R2 + V_parallel / R3) * 1000:.4f} mA')

<a id="nodal"></a>

$$
\text{节点电压法 (Nodal Analysis): 把 KCL 写成线性方程组} \\ \quad \\
\text{1. 选一个参考节点 (地 Ground) 电压为 0, 其他节点电压 } v_1, \cdots, v_n \text{ 为未知数} \\
\text{2. 对每个节点 k 写 KCL: 从 k 流出到相邻节点 j 的电流 } = \frac{v_k - v_j}{R_{kj}} = g_{kj}(v_k - v_j) \\
\sum_{j} g_{kj} (v_k - v_j) = i_k \quad (i_k \text{: 外部电流源流入节点 k 的电流}) \\
v_k \underbrace{\sum_j g_{kj}}_{\text{接在 k 上的电导之和}} - \sum_{j \neq k} g_{kj} v_j = i_k \\ \quad \\
\text{3. 写成矩阵形式 } G v = i \text{, 电导矩阵 (Conductance Matrix):} \\
G_{kk} = \sum (\text{接在节点 k 上的所有电导, 包括接地的}), \quad G_{kj} = -g_{kj} \ (k \neq j) \\
\text{性质: G 对称 (} g_{kj} = g_{jk} \text{), 严格对角占优, 是正定矩阵, 所以一定可逆, 解唯一} \\
\text{(物理意义: } v^T G v = \sum \text{电导} \times \text{电压差}^2 \text{ = 电阻消耗的总功率 > 0)} \\ \quad \\
\text{电压源 + 串联电阻可以等效成电流源 + 并联电阻 (诺顿等效): } V_s, R \to I = \frac{V_s}{R}, \ g = \frac{1}{R} \text{ 接地} \\ \quad \\
\text{例: } V_s \xrightarrow{R_1} \text{节点 1}, \ R_2: 1 \to \text{地}, \ R_3: 1 \to 2, \ R_4: 2 \to \text{地}, \ R_5: 2 \to 3, \ R_6: 3 \to \text{地}, \ I_s \to \text{节点 3} \\
\begin{bmatrix}
g_1 + g_2 + g_3 & -g_3 & 0 \\
-g_3 & g_3 + g_4 + g_5 & -g_5 \\
0 & -g_5 & g_5 + g_6
\end{bmatrix}
\begin{bmatrix} v_1 \\ v_2 \\ v_3 \end{bmatrix}
=
\begin{bmatrix} V_s g_1 \\ 0 \\ I_s \end{bmatrix}
$$

数学基础: [矩阵的逆与线性方程组](../math_basics/linear_algebra.ipynb#inverse) · [正定矩阵](../math_basics/linear_algebra.ipynb#positive-definite) · [对称矩阵](../math_basics/linear_algebra.ipynb#symmetric-matrix)

In [ ]:
def gauss_elimination(A, b):
    # 高斯消元 (部分主元) + 回代
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)
    for col in range(n):
        pivot = col + np.argmax(np.abs(A[col:, col]))
        A[[col, pivot]], b[[col, pivot]] = A[[pivot, col]], b[[pivot, col]]
        for row in range(col + 1, n):
            factor = A[row, col] / A[col, col]
            A[row, col:] -= factor * A[col, col:]
            b[row] -= factor * b[col]
    x = np.zeros(n)
    for row in range(n - 1, -1, -1):
        x[row] = (b[row] - A[row, row + 1:] @ x[row + 1:]) / A[row, row]
    return x

Vs, Is = 10.0, 2e-3
R1, R2, R3, R4, R5, R6 = 1e3, 2e3, 1e3, 3e3, 2e3, 1e3
g1, g2, g3, g4, g5, g6 = [1 / r for r in (R1, R2, R3, R4, R5, R6)]

G = np.array([[g1 + g2 + g3, -g3, 0],
              [-g3, g3 + g4 + g5, -g5],
              [0, -g5, g5 + g6]])
i_vec = np.array([Vs * g1, 0, Is])

v_gauss = gauss_elimination(G, i_vec)
v_numpy = np.linalg.solve(G, i_vec)
print(f'节点电压 (高斯消元) = {v_gauss} V')
print(f'节点电压 (numpy)    = {v_numpy} V, 一致? {np.allclose(v_gauss, v_numpy)}')
print(f'G 对称? {np.allclose(G, G.T)}, G 的特征值 (mS) = {np.linalg.eigvalsh(G) * 1000} (全部 > 0, 正定)')

# 回代检查每个节点的 KCL: 流出电流之和 = 流入的源电流
v1, v2, v3 = v_numpy
kcl1 = (v1 - Vs) / R1 + v1 / R2 + (v1 - v2) / R3
kcl2 = (v2 - v1) / R3 + v2 / R4 + (v2 - v3) / R5
kcl3 = (v3 - v2) / R5 + v3 / R6 - Is
print(f'KCL 残差 = {[f"{x:.1e}" for x in (kcl1, kcl2, kcl3)]} A')

# 功率守恒: 电源提供的功率 = 所有电阻消耗的功率
P_resistors = ((Vs - v1)**2 / R1 + v1**2 / R2 + (v1 - v2)**2 / R3 + v2**2 / R4 + (v2 - v3)**2 / R5 + v3**2 / R6)
P_sources = Vs * (Vs - v1) / R1 + Is * v3
print(f'电阻消耗功率 = {P_resistors * 1000:.6f} mW, 电源提供功率 = {P_sources * 1000:.6f} mW')

<a id="mesh"></a>

$$
\text{网孔电流法 (Mesh Analysis): 把 KVL 写成线性方程组 (和节点法对偶)} \\ \quad \\
\text{1. 给每个网孔 (不包含其他回路的最小回路) 一个假想的环流 } i_1, i_2, \cdots \\
\text{2. 一个电阻如果被两个网孔共用, 真实电流 = 两个网孔电流之差 (KCL 自动满足)} \\
\text{3. 对每个网孔写 KVL} \\ \quad \\
\text{例 (梯形网络 ladder): } V_s - R_1 - a, \ R_2: a \to \text{地}, \ R_3: a \to b, \ R_4: b \to \text{地} \\
\text{网孔 1 (} V_s, R_1, R_2 \text{): } R_1 i_1 + R_2 (i_1 - i_2) = V_s \\
\text{网孔 2 (} R_2, R_3, R_4 \text{): } R_2 (i_2 - i_1) + R_3 i_2 + R_4 i_2 = 0 \\
\begin{bmatrix} R_1 + R_2 & -R_2 \\ -R_2 & R_2 + R_3 + R_4 \end{bmatrix}
\begin{bmatrix} i_1 \\ i_2 \end{bmatrix} = \begin{bmatrix} V_s \\ 0 \end{bmatrix} \\
\text{电阻矩阵同样对称正定: 对角线是网孔内电阻之和, 非对角线是共用电阻的负值}
$$

In [ ]:
Vs_l = 12.0
Ra, Rb, Rc, Rd = 1e3, 2e3, 1.5e3, 3e3   # R1, R2, R3, R4 of the ladder

# 网孔法
R_mesh = np.array([[Ra + Rb, -Rb],
                   [-Rb, Rb + Rc + Rd]])
i1, i2 = np.linalg.solve(R_mesh, [Vs_l, 0])
v_a_mesh = (i1 - i2) * Rb
v_b_mesh = i2 * Rd

# 节点法 (同一个电路)
G_ladder = np.array([[1 / Ra + 1 / Rb + 1 / Rc, -1 / Rc],
                     [-1 / Rc, 1 / Rc + 1 / Rd]])
v_a_nodal, v_b_nodal = np.linalg.solve(G_ladder, [Vs_l / Ra, 0])

print(f'网孔电流: i1 = {i1 * 1000:.6f} mA, i2 = {i2 * 1000:.6f} mA')
print(f'节点 a: 网孔法 = {v_a_mesh:.6f} V, 节点法 = {v_a_nodal:.6f} V')
print(f'节点 b: 网孔法 = {v_b_mesh:.6f} V, 节点法 = {v_b_nodal:.6f} V')

<a id="thevenin"></a>

$$
\text{戴维南定理 (Thevenin's Theorem): 任何由电阻和独立源组成的线性二端网络, 对外等效于一个电压源串联一个电阻} \\ \quad \\
\text{推导: 在端口接一个外部电流源 I (流出网络), 求端口电压 V} \\
\text{由叠加原理 (Superposition, 线性方程组 } Gv = i \text{ 的解是右端项的线性函数): V 是 I 和内部各个源的线性组合} \\
V = \underbrace{c_0}_{\text{内部源的贡献}} - \underbrace{c_1}_{\text{比例系数}} I \quad (\text{端口的 V-I 关系是一条直线}) \\
\text{令 I = 0 (开路 Open Circuit): } c_0 = V_{oc} \equiv V_{th} \\
\text{令 V = 0 (短路 Short Circuit): } 0 = V_{th} - c_1 I_{sc} \implies c_1 = \frac{V_{th}}{I_{sc}} \equiv R_{th} \\
V = V_{th} - R_{th} I \text{, 这正是电压源 } V_{th} \text{ 串联电阻 } R_{th} \text{ 的端口关系} \\ \quad \\
R_{th} \text{ 的另一种求法: 把内部独立源置零 (电压源短路, 电流源开路), 从端口看进去的等效电阻} \\
\text{例: 梯形网络把 } R_4 \text{ 当作负载, 端口在 b 和地之间} \\
V_{th} = V_s \frac{R_2}{R_1 + R_2} \ (\text{开路时 } R_3 \text{ 没有电流, 分压}), \quad R_{th} = (R_1 \parallel R_2) + R_3
$$

In [ ]:
def port_voltage(R_load):
    # 解原电路 (负载 R_load 接在 b 和地之间), 返回负载电压
    G_l = np.array([[1 / Ra + 1 / Rb + 1 / Rc, -1 / Rc],
                    [-1 / Rc, 1 / Rc + 1 / R_load]])
    return np.linalg.solve(G_l, [Vs_l / Ra, 0])[1]

# 开路电压 (负载电阻无穷大) 和短路电流 (负载电阻趋于 0)
V_oc = port_voltage(1e15)
I_sc = port_voltage(1e-9) / 1e-9
R_th = V_oc / I_sc
print(f'数值: V_oc = {V_oc:.6f} V, I_sc = {I_sc * 1000:.6f} mA, R_th = V_oc / I_sc = {R_th:.4f} ohm')
print(f'推导: V_th = {Vs_l * Rb / (Ra + Rb):.6f} V, R_th = R1||R2 + R3 = {Ra * Rb / (Ra + Rb) + Rc:.4f} ohm')

# 对各种负载, 原电路和戴维南等效电路给出相同的负载电压
for R_load in [100, 1e3, 3e3, 1e4]:
    print(f'R_load = {R_load:>7.0f}: 原电路 V = {port_voltage(R_load):.6f}, 戴维南 V = {V_oc * R_load / (R_th + R_load):.6f}')

<a id="max-power"></a>

$$
\text{最大功率传输 (Maximum Power Transfer): 负载电阻取多少时, 负载得到的功率最大?} \\ \quad \\
\text{用戴维南等效电路: } I = \frac{V_{th}}{R_{th} + R_L} \implies P_L = I^2 R_L = \frac{V_{th}^2 R_L}{(R_{th} + R_L)^2} \\
\text{对 } R_L \text{ 求导 (商法则):} \\
\frac{dP_L}{dR_L} = V_{th}^2 \frac{(R_{th} + R_L)^2 - R_L \cdot 2 (R_{th} + R_L)}{(R_{th} + R_L)^4} = V_{th}^2 \frac{R_{th} - R_L}{(R_{th} + R_L)^3} \\
\text{令导数为 0: } R_L = R_{th} \text{ (阻抗匹配 Impedance Matching)} \\
R_L < R_{th} \text{ 时导数 > 0 (增加), } R_L > R_{th} \text{ 时导数 < 0 (减少), 所以是极大值} \\
P_{max} = \frac{V_{th}^2 R_{th}}{(2 R_{th})^2} = \frac{V_{th}^2}{4 R_{th}} \\ \quad \\
\text{效率: } \eta = \frac{P_L}{P_{total}} = \frac{I^2 R_L}{I^2 (R_{th} + R_L)} = \frac{R_L}{R_{th} + R_L} = 50\% \text{ (最大功率时一半能量消耗在内阻上)} \\
\text{所以电网追求效率 (} R_L \gg R_{th} \text{), 而信号传输 (天线, 音响) 追求最大功率 (} R_L = R_{th} \text{)}
$$

数学基础: [求导法则 (商法则)](../math_basics/calculus.ipynb#derivative-rules) · [极值的一阶 / 二阶条件](../math_basics/calculus.ipynb#extremum)

In [ ]:
R_L = np.linspace(10, 10000, 2000)
P_L = V_oc**2 * R_L / (R_th + R_L)**2
P_func = lambda r: V_oc**2 * r / (R_th + r)**2
dP_formula = lambda r: V_oc**2 * (R_th - r) / (R_th + r)**3
for r in [500.0, R_th, 5000.0]:
    dP_numeric = (P_func(r + 1e-3) - P_func(r - 1e-3)) / 2e-3
    print(f'R_L = {r:8.2f}: 导数公式 = {dP_formula(r):+.6e}, 中心差分 = {dP_numeric:+.6e}')

res = minimize_scalar(lambda r: -V_oc**2 * r / (R_th + r)**2, bounds=(1, 1e5), method='bounded')
print(f'scipy 最优 R_L = {res.x:.4f} ohm, R_th = {R_th:.4f} ohm')
print(f'P_max: 数值 = {-res.fun * 1000:.6f} mW, V_th^2 / (4 R_th) = {V_oc**2 / (4 * R_th) * 1000:.6f} mW')
print(f'在原电路中直接验证: R_L = R_th 时负载功率 = {port_voltage(R_th)**2 / R_th * 1000:.6f} mW')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].plot(R_L, P_L * 1000, label='P_L (mW)')
axs[0].axvline(R_th, color='r', linestyle='--', label='R_L = R_th')
axs[0].set_xlabel('R_L (ohm)')
axs[1].plot(R_L, R_L / (R_th + R_L), label='efficiency')
axs[1].axvline(R_th, color='r', linestyle='--', label='50% at R_th')
axs[1].set_xlabel('R_L (ohm)')
plot_nicely()
plt.show()